In [0]:
--list catlogs
SHOW CATALOGS like '*d*'

--
spark.catalog.listCatalogs()
--list schemas
SHOW SCHEMAS
spark.catalog.tableExists( )

In [0]:
show schemas in dev like ''

In [0]:
--CREATE TEMP VIEW AND PAERMANET VIEW
CREATE OR REPLACE TEMP VIEW temp_view AS SELECT * FROM VALUES (1, 'a'), (2, 'b') AS tab(id, name);
SELECT * FROM temp_view;




CREATE OR REPLACE VIEW dev.bronze.temp_view AS SELECT * FROM VALUES (1, 'a'), (2, 'b') AS tab(  id, name);
SELECT * FROM dev.bronze.temp_view

In [0]:
-- Create sample sales table in dev.bronze
CREATE OR REPLACE TABLE dev.bronze.sales (
  sale_id INT,
  product_name STRING,
  quantity INT,
  sale_amount DOUBLE,
  sale_date DATE
);

-- Insert around 10 random values into dev.bronze.sales
INSERT INTO dev.bronze.sales
VALUES
  (1, 'Widget', 5, 49.99, DATE('2026-06-01')),
  (2, 'Gadget', 2, 19.99, DATE('2026-06-02')),
  (3, 'Thingamajig', 7, 79.99, DATE('2026-06-03')),
  (4, 'Doodad', 3, 29.99, DATE('2026-06-04')),
  (5, 'Contraption', 1, 99.99, DATE('2026-06-05')),
  (6, 'Device', 4, 39.99, DATE('2026-06-06')),
  (7, 'Apparatus', 6, 59.99, DATE('2026-06-07')),
  (8, 'Instrument', 8, 89.99, DATE('2026-06-08')),
  (9, 'Tool', 2, 24.99, DATE('2026-06-09')),
  (10, 'Machine', 5, 109.99, DATE('2026-06-10'));

# Databricks Table Copy & Clone Notes

## 1. CTAS (Create Table As Select)

### Create a Physical Copy of Data

```sql
CREATE TABLE dev.bronze.sales_copy
AS
SELECT *
FROM dev.bronze.sales;
```

### Characteristics

* Creates a completely new table.
* Copies data from source table.
* Creates a new Delta table.
* Source and target become independent.
* Changes in source do not affect target.
* Changes in target do not affect source.

### Use Cases

* Data backup
* Data migration
* Creating test environments
* Creating filtered datasets

Example:

```sql
CREATE TABLE dev.bronze.sales_2025
AS
SELECT *
FROM dev.bronze.sales
WHERE year = 2025;
```

---

# 2. Deep Clone

### Syntax

```sql
CREATE TABLE dev.bronze.sales_deep_clone
DEEP CLONE dev.bronze.sales;
```

### What Gets Copied?

✔ Metadata

✔ Data Files

✔ Delta Transaction Log

✔ Table History

✔ Schema

### Characteristics

* Full physical copy of source table.
* Independent table after clone.
* Source can be deleted without affecting clone.
* Requires additional storage.

### Use Cases

* Disaster recovery
* Production backup
* Cross-environment migration
* Long-term archival

### Storage

Consumes additional storage because data files are copied.

---

# 3. Shallow Clone

### Syntax

```sql
CREATE TABLE dev.bronze.sales_shallow_clone
SHALLOW CLONE dev.bronze.sales;
```

### What Gets Copied?

✔ Metadata

✔ Delta Log References

❌ Data Files

### Characteristics

* Does not copy underlying data files.
* References source table data files.
* Very fast operation.
* Minimal storage consumption.

### Use Cases

* Development environments
* Testing
* Quick experimentation
* Data validation

### Storage

Consumes very little storage.

---

# Deep Clone vs Shallow Clone

| Feature               | Deep Clone | Shallow Clone |
| --------------------- | ---------- | ------------- |
| Metadata Copied       | Yes        | Yes           |
| Data Copied           | Yes        | No            |
| Delta Log Copied      | Yes        | Yes           |
| Storage Usage         | High       | Very Low      |
| Clone Speed           | Slower     | Very Fast     |
| Independent of Source | Yes        | No            |
| Suitable for Backup   | Yes        | No            |
| Suitable for Dev/Test | Yes        | Yes           |

---

# Important Interview Question

## What happens if source table is deleted?

### Deep Clone

```sql
DROP TABLE dev.bronze.sales;
```

Deep clone still works because:

* Data files were copied.
* Clone is fully independent.

### Shallow Clone

```sql
DROP TABLE dev.bronze.sales;
```

Potential issue:

* Shallow clone references source data files.
* If source files are removed (VACUUM or storage cleanup), shallow clone may become unusable.

---

# Clone History

View clone operation:

```sql
DESCRIBE HISTORY dev.bronze.sales_deep_clone;
```

or

```sql
DESCRIBE HISTORY dev.bronze.sales_shallow_clone;
```

---

# Clone with Version

Create clone from specific version.

### Deep Clone

```sql
CREATE TABLE dev.bronze.sales_backup
DEEP CLONE dev.bronze.sales VERSION AS OF 25;
```

### Shallow Clone

```sql
CREATE TABLE dev.bronze.sales_test
SHALLOW CLONE dev.bronze.sales VERSION AS OF 25;
```

Useful for point-in-time recovery.

---

# CTAS vs Deep Clone vs Shallow Clone

| Feature                 | CTAS   | Deep Clone | Shallow Clone |
| ----------------------- | ------ | ---------- | ------------- |
| Copies Data             | Yes    | Yes        | No            |
| Copies Metadata         | No     | Yes        | Yes           |
| Copies Table Properties | No     | Yes        | Yes           |
| Copies History          | No     | Yes        | Yes           |
| Independent Table       | Yes    | Yes        | No            |
| Storage Cost            | High   | High       | Low           |
| Execution Speed         | Medium | Medium     | Fast          |

---

# Best Practice

### Use CTAS

When:

* Transforming data
* Filtering data
* Creating reporting tables

### Use Deep Clone

When:

* Backup required
* Disaster recovery
* Production migration
* Long-term retention

### Use Shallow Clone

When:

* Development work
* Testing
* Temporary environments
* Quick data exploration

---

# Quick Memory Trick

CTAS
= Copy Data Only

Deep Clone
= Copy Metadata + Data

Shallow Clone
= Copy Metadata Only (References Original Data)


In [0]:
-- create copy of the table
--ctas for delta table 

create table dev.bronze.sales_copy as select * from dev.bronze.sales;
--deep clone - metadata and date
create table dev.bronze.sales_deep_clone deep clone  dev.bronze.sales;
--shallow clone - metadata only
create table dev.bronze.sales_shallow_clone shallow clone   dev.bronze.sales;





# Clone Behavior After Source Changes

## CTAS

```sql
CREATE TABLE sales_copy AS
SELECT * FROM sales;
```

* Independent copy.
* New inserts in source → NOT reflected.
* Updates in source → NOT reflected.
* Deletes in source → NOT reflected.
* DROP source table → copy unaffected.
* VACUUM source → copy unaffected.

---

## Deep Clone

```sql
CREATE TABLE sales_deep_clone
DEEP CLONE sales;
```

* Independent copy.
* New inserts in source → NOT reflected.
* Updates in source → NOT reflected.
* Deletes in source → NOT reflected.
* DROP source table → clone unaffected.
* VACUUM source → clone unaffected.
* Safe for backup and disaster recovery.

---

## Shallow Clone

```sql
CREATE TABLE sales_shallow_clone
SHALLOW CLONE sales;
```

* Metadata copied.
* References source data files.
* New inserts in source after clone → NOT reflected.
* Updates in source after clone → NOT reflected.
* Deletes in source after clone → NOT reflected.
* DROP source table → clone may still work if files exist.
* VACUUM source → DANGEROUS.
* If VACUUM removes referenced files, shallow clone can become unusable.

---

# Interview One-Liner

CTAS = Copy Data

Deep Clone = Copy Data + Metadata (Independent)

Shallow Clone = Copy Metadata, Reuse Data Files (Dependent on Source Files)
